# Create FLUT

The test requires a non-reactive FLUT with variable enthalpy/temperature and constant mixture.

In [ ]:
import pyFLUT
import cantera as ct
import numpy as np

Load mechanism (included in cantera)

In [ ]:
gas = ct.Solution('ch4_smooke_EKT.yaml')

In [ ]:
gas.species_names

### Create manifold

We create a manifold whose temperature is wider than the expected (particle and gas @450 K)

In [ ]:
# Air
air = {'O2': 0.233, 'N2': 0.767}

fuels = { 
        'fuel1': {'CH2O': 0.4, 'CH3O': 0.4, 'CH3': 0.2}, 
        'fuel2': {'CH4': 1}, 
        'fuel3': {'CO': 0.3474, 'N2': 0.6526} 
        }

# Temperatures
Ts = np.linspace(300,500,11)

# Fuel blend
Zs = np.linspace(0,0.5,26)
blending =      {
                'Y': np.linspace(0,1,6),
                'YY': np.linspace(0,1,6),
                }

parameters = tuple( [Zs] + [Ys for Ys in blending.values()] + [Ts] )

# Generate meshgrid
combination_tuple = np.meshgrid(*parameters , indexing='ij')

# Combine into 2xn shape
combination_array = np.concatenate(tuple([x[...,np.newaxis] for x in combination_tuple]),axis=len(combination_tuple))

combination_array.shape

Create solution array

In [ ]:
sol_arr = ct.SolutionArray(gas,combination_array[...,0].shape)

In [ ]:
species_array = np.zeros(tuple(list(combination_array.shape[:-1]) + [gas.n_species]))
for spec, Y in air.items():
    species_array[...,gas.species_index(spec)] += Y * (1 - combination_array[...,0])
for i, (fuel, specs) in enumerate(fuels.items()):
    for spec, Y in specs.items():
        species_array[...,gas.species_index(spec)] +=  Y * np.prod(combination_array[...,:i+1],axis=-1 ) * (( 1 - combination_array[...,i+1]) if i+3 < len(combination_array.shape) else 1)

In [ ]:
sol_arr.TPY = combination_array[...,-1], ct.one_atm, species_array

### Build flut from the data

In [ ]:
data = np.concatenate((sol_arr.T[...,np.newaxis],sol_arr.P[...,np.newaxis],sol_arr.density[...,np.newaxis],sol_arr.enthalpy_mass[...,np.newaxis],species_array),axis=len(combination_array.shape)-1)

In [ ]:
import collections

input_dict = {'Z':Zs}
input_dict.update(blending)
input_dict.update({'T':Ts})

input_dict = collections.OrderedDict(input_dict)
output_list = [(name,i) for i, name in enumerate(['T','p','rho','ha'])]
output_list = output_list + [(name, i + len(output_list)) for i, name in enumerate(gas.species_names)]
output_dict = collections.OrderedDict(output_list)

flut = pyFLUT.Flut(data,input_dict=input_dict,output_dict=output_dict)
flut

In [ ]:
var = 'CO'
np.max(flut[var]), np.min(flut[var])

In [ ]:
flut.calc_non_dimensional_quantity(along_variable='T',quantity='ha')
flut = flut.map_variables(from_inp='T',to_inp='hanorm')
flut

Calc additionally needed variables and rename (maybe this can be adapted in the future)

In [ ]:
flut.calc_with_cantera(['rho', 'cp', 'lambda', 'visc','alpha'],gas=gas)
flut = flut.rename_variables(rename_dict={'visc':'mu'})

In [ ]:
flut.output_dict

In [ ]:
output_variables = ['Z','T','p','rho','ha','hanorm','ha_max','ha_min','cp','lambda','mu','alpha']
# Add air and all fuel species
output_variables = output_variables + list(air.keys()) + [item for spec_list in [list(fuel.keys()) for fuel in fuels.values()] for item in spec_list]
flut.export_to_openfoam('Z_Y_YY_hanorm_non_reactive.h5',cantera_file='ch4_smooke_EKT.yaml',output_variables=output_variables)

### Check 

In [ ]:
read_flut = pyFLUT.Flut.read_h5('Z_Y_YY_hanorm_non_reactive.h5')

In [ ]:
read_flut.output_dict

In [ ]:
read_flut